In [ ]:
import numpy as np
from skimage.measure import regionprops

# 1. Your Mock Data or Loaded Mask
mask = np.zeros((100, 100, 100))
mask[40:60, 40:60, 40:60] = 1

# 2. Physics Spacing (mm)
spacing = (1.0, 1.0, 1.0)

# 3. Extract Properties
props = regionprops(mask.astype(int))

# 4. Calculate Volume (Safe & Error-Free)
if len(props) > 0:
    obj = props[0]
    voxel_vol = np.prod(spacing)
    volume_mm3 = obj.area * voxel_vol

    print(f"Volume: {volume_mm3:.2f} mm3")
else:
    print("No object found in mask!")

Volume: 8000.00 mm3


In [ ]:
# 1. Install dependencies first
!pip install SimpleITK numpy pandas

# 2. Install the latest development version of pyradiomics from GitHub
# This version contains the fixes for Python 3.12
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git@master
import radiomics
from radiomics import firstorder, shape, getTestCase
import SimpleITK as sitk
import pandas as pd

print("Radiomics Version:", radiomics.__version__)
print("Import Successful! You can now use PyRadiomics.")
# 2. Get the Sample Data (Brain Tumor Case 'brain1')
# PyRadiomics provides this automatically for testing
imagePath, maskPath = getTestCase('brain1')

# 3. Load the data into SimpleITK
# This format keeps the "Spacing" (the gaps between slices) intact!
image = sitk.ReadImage(imagePath)
mask = sitk.ReadImage(maskPath)

# 4. Initialize the Calculators
# These are the "Toolboxes" we discussed
shape_calc = shape.RadiomicsShape(image, mask)
first_order_calc = firstorder.RadiomicsFirstOrder(image, mask)

# 5. Execute the calculations
# This is where the 'Marching Cubes' and 'Voxel Counting' happen
shape_features = shape_calc.execute()
first_order_features = first_order_calc.execute()

# 6. Combine all results into a clean table
all_data = {**shape_features, **first_order_features}
df = pd.DataFrame.from_dict(all_data, orient='index', columns=['Value'])

# 7. Display the Key Results for Analysis
print("--- RADIOMICS FEATURE EXTRACTION COMPLETE ---")
print(f"Tumor Volume (Mesh): {all_data['MeshVolume']:.2f} mm3")
print(f"Tumor Sphericity: {all_data['Sphericity']:.4f} (1.0 = Perfect Sphere)")
print(f"Surface Area: {all_data['SurfaceArea']:.2f} mm2")
print(f"Average Brightness (Mean): {all_data['Mean']:.2f} HU")

# Show the first 15 features extracted
df.tail(5)

  Cloning https://github.com/AIM-Harvard/pyradiomics.git (to revision master) to /tmp/pip-req-build-rru86t03
  Running command git clone --filter=blob:none --quiet https://github.com/AIM-Harvard/pyradiomics.git /tmp/pip-req-build-rru86t03
  Resolved https://github.com/AIM-Harvard/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Radiomics Version: 3.1.1.dev111+g8ed579383
Import Successful! You can now use PyRadiomics.
--- RADIOMICS FEATURE EXTRACTION COMPLETE ---
Tumor Volume (Mesh): 16147.51 mm3
Tumor Sphericity: 0.4798 (1.0 = Perfect Sphere)
Surface Area: 6438.82 mm2
Average Brightness (Mean): 825.24 HU


,Value
RootMeanSquared,839.9646448180755
Skewness,0.27565085908587594
TotalEnergy,11579797135.314934
Uniformity,0.045156963555862184
Variance,24527.07920837261
